[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C27_Model_Compression_Course/03_fp8_training/03_fp8_training.ipynb)

# 03 · fp8 训练（用 numpy 模拟）

用 7 个 bit 算矩阵乘换两倍吞吐，再用 scaling 把数值喂稳。**从零模拟 fp8、从零实现 scaling**。

**路线**：
1. 从零模拟 fp8 cast（E4M3/E5M2，含舍入与下溢）
2. E4M3 vs E5M2：精度 vs 范围的取舍
3. dynamic scaling：把张量挪进 fp8 甜区
4. **loss scaling**：把下溢的梯度从几千个救回个位数
5. 动态 loss scaling：溢出减半、平稳翻倍
6. 混合精度：fp8 算、fp32 记（master weights 的必要性）
7. ✏️ 练习（fp8 cast / 范围 / loss scaling / 混合精度）
8. 📖 答案 · 🧪 真实激活分布胶囊

> **本课纪律**：每个机制都和全精度对拍。E4M3 误差<E5M2、scaling 降误差、loss scaling 减下溢——都要 assert 验证。

## 1 · 从零模拟 fp8 cast（E4M3 / E5M2）

fp8 = `(-1)^s × 1.尾数 × 2^(指数-bias)`。模拟 cast：按格式的尾数位数把每个数**舍入到最近可表示值**，
超范围 clip，太小则**下溢为 0**。E4M3 范围 ±448、3 尾数位；E5M2 范围 ±57344、2 尾数位。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

FP8 = {
    'e4m3': dict(mbits=3, fmax=448.0,   min_sub=2.0**-9),   # 最小次正规 ~0.00195
    'e5m2': dict(mbits=2, fmax=57344.0, min_sub=2.0**-16),
}

def to_fp8(x, fmt='e4m3'):
    '''把 x 量化到 fp8 可表示值(模拟): 按 mbits 舍入尾数, clip 范围, 下溢为 0。'''
    cfg = FP8[fmt]; mbits, fmax, min_sub = cfg['mbits'], cfg['fmax'], cfg['min_sub']
    x = np.asarray(x, dtype=np.float64)
    sign = np.sign(x); ax = np.abs(x)
    ax = np.clip(ax, 0, fmax)                          # 超上界 -> clip
    big = ax >= min_sub / 2                            # 太小 -> 下溢为 0
    with np.errstate(divide='ignore'):
        e = np.where(big, np.floor(np.log2(np.where(big, ax, 1.0))), 0.0)
    scale = 2.0 ** (e - mbits)                         # 该指数档的尾数步长
    q = np.round(ax / scale) * scale                  # 舍入到最近可表示值
    q = np.where(big, q, 0.0)                          # 下溢清零
    return sign * q

x = np.array([0.0, 0.1, 1.0, 3.14159, 100.0, 500.0, 1e-4])
print('原始  :', x)
print('E4M3  :', to_fp8(x, 'e4m3'))
print('  注意: 500>448 被 clip 到 448; 1e-4 下溢为 0')
# 0 必须精确表示; 范围内的值舍入误差有界
assert to_fp8(np.array([0.0]))[0] == 0.0
assert to_fp8(np.array([500.0]), 'e4m3')[0] == 448.0, '超范围应 clip 到 fmax'
print('✅ fp8 cast 正确：0 精确、超范围 clip、过小下溢')

## 2 · E4M3 vs E5M2：精度 vs 范围

E4M3 尾数多1位 → **精度高**；E5M2 指数多1位 → **范围大**。
用集中分布(像激活)验证 E4M3 更准；用跨度大的分布(像梯度)验证 E5M2 不溢出而 E4M3 会。

In [ ]:
# 集中分布(激活): E4M3 更准
act = rng.standard_normal(20000) * 0.1
err_e4m3 = np.linalg.norm(to_fp8(act,'e4m3') - act) / np.linalg.norm(act)
err_e5m2 = np.linalg.norm(to_fp8(act,'e5m2') - act) / np.linalg.norm(act)
print(f'集中分布(激活): E4M3 相对误差={err_e4m3:.3%}  E5M2={err_e5m2:.3%}')
assert err_e4m3 < err_e5m2, 'E4M3 尾数多, 应更准'

# 跨度大分布(梯度): 含大值 -> E4M3 溢出, E5M2 扛得住
grad = rng.standard_normal(20000) * 0.5
grad[:50] = rng.standard_normal(50) * 2000.0       # 注入大梯度(>448)
n_overflow_e4m3 = (np.abs(to_fp8(grad,'e4m3')) >= 448.0).sum()
n_overflow_e5m2 = (np.abs(to_fp8(grad,'e5m2')) >= 57344.0).sum()
print(f'跨度大分布(梯度): E4M3 顶到上界(溢出)的数={n_overflow_e4m3}  E5M2={n_overflow_e5m2}')
assert n_overflow_e4m3 > n_overflow_e5m2, 'E4M3 范围小, 大梯度更易溢出'
print('✅ E4M3 精度高(前向) / E5M2 范围大(梯度) —— 各司其职')

## 3 · dynamic scaling：把张量挪进甜区

直接 cast 一个数值偏小的张量到 fp8 会大量下溢。dynamic scaling：`s=fp8_max/amax`，
让最大元素顶到 fp8 上界、用满范围，cast 后再除回 s。对比有无 scaling 的误差。

In [ ]:
def fp8_with_scaling(x, fmt='e4m3', use_scale=True):
    fmax = FP8[fmt]['fmax']
    if use_scale:
        amax = np.abs(x).max()
        s = fmax / amax if amax > 0 else 1.0          # 让 amax 顶到 fp8 上界
        return to_fp8(x * s, fmt) / s                 # cast 后除回
    return to_fp8(x, fmt)

# 一个数值偏小的张量(很多值接近 fp8 下溢区)
x = rng.standard_normal(20000) * 0.003
err_noscale = np.linalg.norm(fp8_with_scaling(x,'e4m3',False) - x) / np.linalg.norm(x)
err_scaled  = np.linalg.norm(fp8_with_scaling(x,'e4m3',True)  - x) / np.linalg.norm(x)
n_zero_noscale = ((fp8_with_scaling(x,'e4m3',False)==0) & (x!=0)).sum()
n_zero_scaled  = ((fp8_with_scaling(x,'e4m3',True) ==0) & (x!=0)).sum()
print(f'无 scaling: 相对误差={err_noscale:.2%}  下溢为0的元素={n_zero_noscale}')
print(f'有 scaling: 相对误差={err_scaled:.2%}  下溢为0的元素={n_zero_scaled}')
assert err_scaled < err_noscale, 'scaling 应降低误差'
assert n_zero_scaled < n_zero_noscale, 'scaling 应减少下溢'
print('✅ dynamic scaling 把张量挪进甜区，误差和下溢都大降')

## 4 · loss scaling：救回下溢的梯度

很多梯度太小、cast 到 fp8 下溢为 0 → 权重学不动。loss scaling：反向前把 loss 乘 `L_scale`，
所有梯度等比放大、躲过下溢；更新前除回。这是最经典、最必须的 fp8 训练技巧。

In [ ]:
# 模拟一批很小的梯度(训练后期/深层常见)
grad = rng.standard_normal(5000) * 1e-3
grad[np.abs(grad) < 2e-3] *= 0.01                    # 让很多梯度变得极小
print(f'低于 E4M3 最小次正规(~0.00195)的梯度占比: {(np.abs(grad)<FP8["e4m3"]["min_sub"]).mean():.1%}')

# 不用 loss scaling: 直接 cast 梯度到 fp8
g_noscale = to_fp8(grad, 'e4m3')
n_underflow_noscale = ((g_noscale == 0) & (grad != 0)).sum()

# 用 loss scaling: 梯度先 ×L_scale 再 cast, 更新前 ÷L_scale
L_scale = 448.0 / np.abs(grad).max()                 # 把最大梯度顶到 fp8 上界
g_scaled = to_fp8(grad * L_scale, 'e4m3') / L_scale
n_underflow_scaled = ((g_scaled == 0) & (grad != 0)).sum()

print(f'不用 loss scaling: 下溢为0的梯度 = {n_underflow_noscale} / {grad.size}')
print(f'用   loss scaling: 下溢为0的梯度 = {n_underflow_scaled} / {grad.size}')
assert n_underflow_scaled < n_underflow_noscale / 10, 'loss scaling 应大幅减少下溢'
print('✅ loss scaling 把下溢的梯度从几千个救回到个位数 —— 训练才学得动')

## 5 · 动态 loss scaling：溢出减半、平稳翻倍

loss scale 太小→梯度下溢；太大→梯度溢出 inf。**动态 loss scaling**：从大值起，
遇 inf/NaN 就减半并跳过这步；连续 N 步正常就翻倍。自动找「尽量大但不溢出」的 scale。

In [ ]:
class DynamicLossScaler:
    def __init__(self, init=2.0**14, factor=2.0, patience=200):
        self.scale = init; self.factor = factor; self.patience = patience
        self.good_steps = 0
    def update(self, grad_has_inf):
        if grad_has_inf:
            self.scale = max(1.0, self.scale / self.factor)   # 溢出: 减半
            self.good_steps = 0
            return False                                       # 跳过这步
        self.good_steps += 1
        if self.good_steps >= self.patience:                  # 连续正常: 翻倍
            self.scale *= self.factor; self.good_steps = 0
        return True                                            # 正常更新

scaler = DynamicLossScaler(init=2.0**20, patience=3)
# 模拟: 一开始 scale 太大导致溢出, scaler 自动减半到安全区
fp8_max = 448.0
grad_amax = 1.0                                       # 梯度最大绝对值
history = []
for step in range(30):
    overflow = (grad_amax * scaler.scale) > fp8_max  # scale 太大 -> 溢出
    applied = scaler.update(overflow)
    history.append((scaler.scale, overflow, applied))
final_scale = scaler.scale
print(f'最终 loss scale = {final_scale:.0f}  (安全上界 = {fp8_max/grad_amax:.0f})')
print(f'前几步是否溢出: {[h[1] for h in history[:8]]}')
assert final_scale * grad_amax <= fp8_max, '收敛后不应再溢出'
assert final_scale > 1.0, '不应一路减到底'
print('✅ 动态 loss scaling 自动收敛到「尽量大但不溢出」的 scale')

## 6 · 混合精度：fp8 算、fp32 记（master weights）

权重更新量通常极小。若权重只有 fp8 精度，小更新会被**大数吃小数**舍掉 → 训练原地踏步。
所以必须 **fp32 主权重**：fp32 累积更新、前向才临时 cast 成 fp8。对比有无 master weights。

In [ ]:
n_steps = 2000
lr_update = 1e-4                                       # 每步极小的更新量

# 不用 master weights: 权重存成 fp8, 每步更新后立刻 cast 回 fp8
w_fp8 = np.array([1.0])
for _ in range(n_steps):
    w_fp8 = to_fp8(w_fp8 + lr_update, 'e4m3')         # 小更新被 fp8 精度吃掉

# 用 master weights: fp32 累积, 仅前向 cast(这里不改 master)
w_master = np.array([1.0])                            # fp32
for _ in range(n_steps):
    w_master = w_master + lr_update                   # fp32 精确累积
    _ = to_fp8(w_master, 'e4m3')                      # 前向用 fp8(不回写 master)

expected = 1.0 + n_steps * lr_update
print(f'期望权重           = {expected:.4f}')
print(f'无 master(fp8存)   = {w_fp8[0]:.4f}  <- 小更新被吃, 没动!')
print(f'有 master(fp32存)  = {w_master[0]:.4f}  <- 正确累积')
assert abs(w_master[0] - expected) < 1e-6, 'master weights 应精确累积'
assert abs(w_fp8[0] - expected) > 0.1, 'fp8 存权重应丢失小更新'
print('✅ master weights(fp32) 必需：否则 fp8 精度吃掉微小更新，训练停滞')

---
## ✏️ 练习 1：从零写 fp8 cast

实现 `my_to_fp8(x, mbits, fmax)`：按 `mbits` 尾数位舍入、`fmax` 范围 clip（暂不处理下溢，简化）。
目标：0 精确、超范围 clip、尾数多则更准。

In [ ]:
def my_to_fp8(x, mbits=3, fmax=448.0):
    # TODO:
    #   sign=sign(x); ax=clip(|x|, 0, fmax)
    #   e = floor(log2(ax))  (ax>0 处; ax==0 处置 0 避免 log)
    #   scale = 2.0**(e - mbits)
    #   q = round(ax/scale)*scale
    #   返回 sign * q  (ax==0 处应得 0)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
x = np.array([0.0, 0.5, 1.0, 3.14, 100.0, 1000.0])
q3 = my_to_fp8(x, mbits=3, fmax=448.0)            # 像 E4M3
q2 = my_to_fp8(x, mbits=2, fmax=448.0)            # 尾数更少
assert q3[0] == 0.0, '0 必须精确'
assert q3[-1] == 448.0, '1000>448 应 clip'
xx = rng.standard_normal(5000)*0.2
e3 = np.linalg.norm(my_to_fp8(xx,3,448)-xx); e2 = np.linalg.norm(my_to_fp8(xx,2,448)-xx)
assert e3 < e2, '尾数位多应更准'
print('✅ 练习 1 通过：fp8 cast 正确，尾数多则精度高')

## ✏️ 练习 2：格式范围

实现 `fp8_max_value(ebits, mbits)`：给指数位、尾数位，返回该 fp8 格式的最大可表示正值。
公式：`(2 - 2^-mbits) × 2^(2^(ebits-1) - 1)`（IEEE 风格，bias=2^(ebits-1)-1，最大正规指数）。

In [ ]:
def fp8_max_value(ebits, mbits):
    # TODO: bias = 2**(ebits-1) - 1; 最大指数 e_max = (2**ebits - 1) - 1 - bias
    #       最大尾数 = 2 - 2**(-mbits); 返回 最大尾数 * 2**e_max
    #   (注: 真实 fp8 对 inf/nan 的编码略有特例, 这里用规整公式即可)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
m_e5m2 = fp8_max_value(5, 2)
m_e4m3 = fp8_max_value(4, 3)
print(f'E5M2 max ≈ {m_e5m2:.0f}   E4M3 max ≈ {m_e4m3:.0f}')
assert m_e5m2 > m_e4m3, 'E5M2 指数多, 范围更大'
assert m_e5m2 > 50000, 'E5M2 max 应是 5 万级'
print('✅ 练习 2 通过：指数位越多，动态范围越大')

## ✏️ 练习 3：loss scaling 减少下溢

实现 `apply_loss_scaling(grad, L_scale, mbits, fmax, min_sub)`：梯度 ×L_scale → cast fp8 → ÷L_scale，
返回还原后的梯度。目标：合适的 L_scale 能显著减少下溢为 0 的梯度数。

In [ ]:
# 先给一个完整 fp8 cast(含下溢)工具, 供练习 3 使用
def to_fp8_full(x, mbits, fmax, min_sub):
    sign=np.sign(x); ax=np.clip(np.abs(x),0,fmax); big=ax>=min_sub/2
    with np.errstate(divide='ignore'):
        e=np.where(big, np.floor(np.log2(np.where(big,ax,1.0))), 0.0)
    q=np.round(ax/2.0**(e-mbits))*2.0**(e-mbits)
    return sign*np.where(big,q,0.0)
print('to_fp8_full 就绪')

In [ ]:
def apply_loss_scaling(grad, L_scale, mbits=3, fmax=448.0, min_sub=2.0**-9):
    # TODO: 返回 to_fp8_full(grad*L_scale, mbits, fmax, min_sub) / L_scale
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
g = rng.standard_normal(4000) * 5e-4                  # 很小的梯度
n_uf_1 = ((apply_loss_scaling(g, 1.0)==0) & (g!=0)).sum()        # 不放大
Ls = 448.0/np.abs(g).max()
n_uf_big = ((apply_loss_scaling(g, Ls)==0) & (g!=0)).sum()       # 放大
print(f'L_scale=1   下溢梯度 = {n_uf_1}')
print(f'L_scale={Ls:.0f} 下溢梯度 = {n_uf_big}')
assert n_uf_big < n_uf_1 / 5, 'loss scaling 应大幅减少下溢'
print('✅ 练习 3 通过：loss scaling 显著减少梯度下溢')

## ✏️ 练习 4：混合精度——master weights

实现 `train_with_master(w0, updates, mbits, fmax)`：用 fp32 主权重累积一串更新，
每步前向 cast 成 fp8（不回写主权重），返回最终主权重。目标：精确等于 `w0 + sum(updates)`。

In [ ]:
def train_with_master(w0, updates, mbits=3, fmax=448.0):
    # TODO:
    #   w = float(w0)  (fp32 主权重)
    #   for u in updates: w = w + u (fp32 累积); _ = my_to_fp8([w],mbits,fmax) (前向用,不回写)
    #   返回 w
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
updates = [1e-4] * 3000
w_final = train_with_master(1.0, updates, mbits=3, fmax=448.0)
expected = 1.0 + sum(updates)
assert abs(w_final - expected) < 1e-6, f'master 应精确累积: {w_final} vs {expected}'
print(f'✅ 练习 4 通过：master weights 精确累积到 {w_final:.4f} (期望 {expected:.4f})')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def my_to_fp8(x, mbits=3, fmax=448.0):
    x = np.asarray(x, float); sign = np.sign(x); ax = np.clip(np.abs(x), 0, fmax)
    pos = ax > 0
    with np.errstate(divide='ignore'):
        e = np.where(pos, np.floor(np.log2(np.where(pos, ax, 1.0))), 0.0)
    scale = 2.0 ** (e - mbits)
    q = np.round(ax / scale) * scale
    return sign * np.where(pos, q, 0.0)

In [ ]:
# 练习 2 参考答案
def fp8_max_value(ebits, mbits):
    bias = 2**(ebits-1) - 1
    e_max = (2**ebits - 1) - 1 - bias
    return (2 - 2.0**(-mbits)) * 2.0**e_max

In [ ]:
# 练习 3 参考答案
def apply_loss_scaling(grad, L_scale, mbits=3, fmax=448.0, min_sub=2.0**-9):
    return to_fp8_full(grad * L_scale, mbits, fmax, min_sub) / L_scale

In [ ]:
# 练习 4 参考答案
def train_with_master(w0, updates, mbits=3, fmax=448.0):
    w = float(w0)
    for u in updates:
        w = w + u
        _ = my_to_fp8(np.array([w]), mbits, fmax)
    return w

---
## 🧪 真实数据胶囊：在真实 GPT-2 激活上做 fp8 + scaling

用**真实 GPT-2** 一层的激活统计，验证 dynamic scaling 让 fp8 误差大降。
**联网失败自动回退**到统计匹配的合成激活（同分布），结论不变。

In [ ]:
def load_gpt2_activation():
    '''真实 GPT-2 在一小段文本上的某层激活; 失败回退合成。'''
    try:
        import torch
        from transformers import GPT2Model, GPT2Tokenizer
        tok = GPT2Tokenizer.from_pretrained('gpt2')
        m = GPT2Model.from_pretrained('gpt2').eval()
        ids = tok('Model compression makes large models cheaper to serve.', return_tensors='pt')
        with torch.no_grad():
            h = m(**ids, output_hidden_states=True).hidden_states[6]
        a = h.detach().numpy().reshape(-1).astype(np.float64)
        print(f'[真实 GPT-2] 第6层激活 {a.shape}, std={a.std():.3f}')
        return a
    except Exception as e:
        print(f'[回退合成] ({type(e).__name__}) 用统计匹配合成激活')
        rng2 = np.random.default_rng(11)
        a = rng2.standard_normal(8000) * 0.02          # GPT-2 中层激活量级偏小
        a[:20] *= 30                                    # 少量大激活(离群)
        return a

act = load_gpt2_activation()
print(f'激活 amax={np.abs(act).max():.4f}  (远小于 fp8 上界 -> 不 scaling 会大量下溢)')

In [ ]:
def fp8_quality(act, use_scale):
    # TODO: 用 E4M3(mbits=3,fmax=448,min_sub=2**-9) cast act,
    #       use_scale 时先 ×(448/amax) 再 cast 再 ÷回; 返回相对误差 ‖q-act‖/‖act‖
    raise NotImplementedError

In [ ]:
# 自测
err_no = fp8_quality(act, use_scale=False)
err_yes = fp8_quality(act, use_scale=True)
print(f'真实(或合成)GPT-2 激活 E4M3:')
print(f'  无 scaling 相对误差 = {err_no:.2%}')
print(f'  有 scaling 相对误差 = {err_yes:.2%}')
assert err_yes < err_no, 'scaling 应降低 fp8 误差'
print('✅ 胶囊通过：真实激活上 dynamic scaling 让 fp8 误差大降 —— 这就是 fp8 训练必配 scaling 的原因')

In [ ]:
# 📖 胶囊参考答案
def fp8_quality(act, use_scale):
    fmax, min_sub = 448.0, 2.0**-9
    if use_scale:
        s = fmax / np.abs(act).max()
        q = to_fp8_full(act * s, 3, fmax, min_sub) / s
    else:
        q = to_fp8_full(act, 3, fmax, min_sub)
    return np.linalg.norm(q - act) / np.linalg.norm(act)

### 小结
- **fp8 = 7 个 bit 抢吞吐**(张量核心约 2× bf16)，但范围精度都不够 → 必须配 scaling。
- **两种格式**：E4M3(3尾数,范围±448,精度高)用于**前向**；E5M2(2尾数,范围±57344,范围大)用于**梯度**。
- **scaling**：dynamic(`s=fp8_max/amax`)把张量挪进甜区；delayed(用 amax 历史)省扫描开销。
- **loss scaling**：反向前 loss ×L_scale 救下溢梯度、更新前除回；动态调整(溢出减半、平稳翻倍)。
- **混合精度口诀 = fp8 算、fp32 记**：GEMM 用 fp8，累加/master weights/优化器状态用 fp32。否则小更新被吃、训练停滞。
- fp8 是**训练加速手段**不是模型格式：训完仍以 bf16 存/部署，推理再单独做 int4 量化(模块01/02)。

下一站：**模块 04 · 知识蒸馏** —— 不压单个张量，而是把大模型的知识压进一个小模型。